# 📑 Job Search Analytics Dashboard

> **Description:** An automated report analyzing my job application funnel.  
> Data is dynamically sourced from Google Sheets and updated on a weekly basis.

## 🎯 Objective
The purpose of this project is to analyze my job search process as a data pipeline and extract actionable insights about response rates, hiring trends, and application outcomes.

## 📊 What This Dashboard Measures
- Number of applications over time
- Response rate and interview rate
- Distribution of application statuses
- Average and median response time
- Senior vs Non-Senior role performance

## 🗂 Dataset Description
The dataset contains real job application records with the following fields:
- Company
- Position
- Date Applied
- Response Date
- Status

The data is updated continuously and reflects a real-world job search process in the European market.

# 🛠 1. Setup & Environment

In [1]:
#@title [⚙️] Configuration & Auth

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [2]:
#@title [📦] Imports

import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# 📥 2. Data Ingestion & ETL

In [3]:
#@title [🔗] Google Sheets Connection

filename = 'job-search-analytics'

sh = gc.open(filename)

worksheet = sh.get_worksheet(0)

rows = worksheet.get_all_values()
df = pd.DataFrame.from_records(rows[1:], columns=rows[0])

sample_data = {
    'Company': ['Company A', 'Company B', 'Company C', 'Company D', 'Company E'],
    'Position': ['Data Analyst', 'Senior Data Analyst', 'Analytics Engineer', 'Revenue Analyst', 'Product Analyst'],
    'date_applied': ['2026-01-02', '2026-01-03', '2026-01-05', '2026-01-06', '2026-01-08'],
    'response_date': ['2026-01-10', None, '2026-01-09', '2026-01-20', None],
    'status': ['Rejected', '', 'Interview', 'Test task', '']
}

df_sample = pd.DataFrame(sample_data)

print("Structure of the dataset (Sample):")
display(df_sample.head())

Structure of the dataset (Sample):


,Company,Position,date_applied,response_date,status
0,Company A,Data Analyst,2026-01-02,2026-01-10,Rejected
1,Company B,Senior Data Analyst,2026-01-03,None,
2,Company C,Analytics Engineer,2026-01-05,2026-01-09,Interview
3,Company D,Revenue Analyst,2026-01-06,2026-01-20,Test task
4,Company E,Product Analyst,2026-01-08,None,


In [4]:
#@title [🧹] Data Cleaning & Preprocessing

# Date fields update (string → date)
df["date_applied"] = pd.to_datetime(df["date_applied"], errors="coerce")
df["response_date"] = pd.to_datetime(df["response_date"], errors="coerce")

# Status check (empty, NaN → "")
df["status"] = df["status"].astype(str).str.strip()
df.loc[df["status"].isin(["", "nan", "None"]), "status"] = ""

# Creating a final status (for analytics)
df["final_status"] = np.where(df["status"] == "", "No response", df["status"])




# Delete lines without a apply date (if any)
df = df[df["date_applied"].notna()].copy()

In [5]:
#@title [🛠️] Feature Engineering

# Set the order of days of the week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Add flag: Was there a response at all?
df["has_response"] = df["response_date"].notna() | (df["status"] != "")


# Response time in days
df["response_time_days"] = (df["response_date"] - df["date_applied"]).dt.days


# Create a continuous week scale (Year-Week)
# This will link October 2025 and January 2026 into one line. For "Job search intensity" heatmap
df['week_commencing'] = df['date_applied'].dt.to_period('W').apply(lambda r: r.start_time)
df['day_of_week'] = df['date_applied'].dt.day_name()


def categorize_status(row):
    if pd.isna(row['response_date']) or row['response_date'] == '':
        return 'No Feedback'
    if row['status'] in ['Interview', 'Test task']:
        return 'Active Interest'
    return 'Rejection'

df['response_status'] = df.apply(categorize_status, axis=1)


df['position_lower'] = df['position'].str.lower()


def get_level(pos):
    if any(word in pos for word in ['senior', 'sr', 'lead', 'II']):
        return 'Senior+'
    if any(word in pos for word in ['junior', 'jr', 'intern', 'trainee']):
        return 'Junior/Intern'
    return 'Middle/Regular'


df['level'] = df['position_lower'].apply(get_level)


In [6]:
#@title [🧮] Metrics Calculation

metrics = {}

metrics['total_apps'] = len(df)
metrics['replied'] = df['has_response'].sum()
metrics['interviews'] = df['status'].isin(['Interview', 'Test task']).sum()
metrics['search_duration'] = (df['date_applied'].max() - df['date_applied'].min()).days
metrics['offers'] = df['status'].isin(['Offer']).sum()

metrics['response_rate'] = (metrics['replied'] / metrics['total_apps'] * 100) if metrics['total_apps'] > 0 else 0
metrics['itw_rate'] = (metrics['interviews'] / metrics['total_apps'] * 100) if metrics['total_apps'] > 0 else 0

metrics['avg_res_time'] = df[df["response_time_days"] >= 0]["response_time_days"].mean()
metrics['med_res_time'] = df[df["response_time_days"] >= 0]["response_time_days"].median()


# 📊 3. Executive Summary (High-Level)

In [18]:
#@title [🧩] Analytics Dashboard Overview

heatmap_data = df.groupby(['week_commencing', 'day_of_week']).size().reset_index(name='count')

heatmap_pivot = heatmap_data.pivot(index='day_of_week', columns='week_commencing', values='count').fillna(0)
heatmap_pivot = heatmap_pivot.reindex(days_order)

# For ease of display on the X-axis, format the column headings in the "Month YY" format
x_labels = [d.strftime('%b %Y') if i == 0 or d.month != heatmap_pivot.columns[i-1].month else ""
            for i, d in enumerate(heatmap_pivot.columns)]

# Create a 3x3 grid
# Use the first two rows and two columns to indicators and a donut
# Dedicate the third row (the entire bottom) to the hitmap
fig = make_subplots(
    rows=3, cols=3,
    row_heights=[0.2, 0.2, 0.6],
    column_widths=[0.4, 0.4, 0.2],
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "pie", "rowspan": 2}],
        [{"type": "indicator"}, {"type": "indicator"}, None],
        [{"type": "xy", "colspan": 3}, None, None]
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.05,
    subplot_titles=("", "", "<b>RESPONSE RATE</b>", "", "", "<b>WEEKLY APPLICATION VOLUME</b>")
)

# --- Indicators ROW 1 ---
fig.add_trace(go.Indicator(
    mode="number", value=metrics['total_apps'],
    title={'text': "<b>Total applications</b>", 'font': {'size': 18, 'color': '#7f8c8d'}},
    number={'font': {'size': 34, 'color': '#7f8c8d'}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=metrics['itw_rate'],
    number={'suffix': "%", 'font': {'size': 34, 'color': '#7f8c8d'}, 'valueformat': ".1f"},
    title={'text': "<b>Conversion to Interview</b>", 'font': {'size': 18, 'color': '#7f8c8d'}}
), row=1, col=2)

# --- Indicators ROW 2 ---
fig.add_trace(go.Indicator(
    mode="number", value=metrics['search_duration'],
    title={'text': "<b>Search days (total)</b>", 'font': {'size': 18, 'color': '#7f8c8d'}},
    number={'font': {'size': 34, 'color': '#7f8c8d'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=metrics['avg_res_time'],
    title={'text': "<b>Average response time (days)</b>", 'font': {'size': 18, 'color': '#7f8c8d'}},
    number={'font': {'size': 34, 'color': '#7f8c8d'}, 'valueformat': ".1f"}
), row=2, col=2)

# --- Donut chart ---
fig.add_trace(go.Pie(
    labels=['Response', 'No response'],
    values=[metrics['replied'], metrics['total_apps'] - metrics['replied']],
    hole=.6,
    marker_colors=['#1D446D', '#5FB1C1'],
    texttemplate="<b>%{label}</b><br>%{percent}",
    insidetextfont=dict(size=16,),
    textinfo='percent+label',
    showlegend=False,
    pull=[0.05, 0]
), row=1, col=3)


# --- Bottom part: HEATMAP ---
fig.add_trace(go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns.astype(str),
    y=heatmap_pivot.index,
    colorscale='Greens',
    showscale=True,  # Add scale on the right
    colorbar=dict(
        title="Apps count",
        thickness=15,
        len=0.5,
        y=0.3, # Move the scale up
        tickvals=[0, heatmap_pivot.values.max()],
        ticktext=["Less", "More"]
    ),
    xgap=3, ygap=3,
    # Custom hover
    hovertemplate="<b>Week: %{x}</b><br>Day: %{y}<br>Applications: <b>%{z}</b><extra></extra>"
), row=3, col=1)


# Subplot Titles
fig.update_annotations(font=dict(size=18), yshift=20)
# yshift=20 moves up the title above the chart by 20 pixels

# Design settings
fig.update_layout(
    height=850,
    template="plotly_white",
    title={'text': "<b>JOB SEARCH ANALYTICS 2025-2026</b>", 'font': {'size': 28}},
    title_x=0.5,
    margin=dict(t=100, b=50, l=50, r=50),
    legend=dict(orientation="h", yanchor="bottom", y=0.55, xanchor="right", x=1)
)

fig.update_yaxes(autorange="reversed", row=3, col=1)

fig.show()

# 🧬 4. Deep Dive: Conversion & Success Factors

In [16]:
#@title [🌪️] Funnel Analysis (classic version)

# Generating data for the funnel
stages = ["<b>Applications</b> 📤", "<b>Replies</b> 📩", "<b>Interviews</b> 🤝"]
values = [metrics['total_apps'], metrics['replied'], metrics['interviews']]


fig = go.Figure(go.Funnel(
    y = stages,
    x = values,
    textinfo = "value+percent initial",
    # Font size for values ​​inside the funnel (values ​​+ %)
    insidetextfont=dict(size=16, color="white"),
    # marker = {"color": ['#636EFA', "#95a5a6", '#00CC96']},
    # connector = {"line": {"color": "white", "width": 3}, "fillcolor": "#f2f2f2"}
    marker = {
        "color": ['#1D446D', '#5FB1C1', '#B34138'],
        "line": {"width": 3, "color": "white"} # The white outline creates a soft effect.
    }
))

fig.update_layout(
    title_text='<b>JOB SEARCH CONVERSION</b>',
    # Font size for stage names (Y-axis)
    yaxis=dict(tickfont=dict(size=16)),
    height=500,
    template="plotly_white"
)

fig.show()



In [15]:
#@title [🌪️] Funnel Analysis (minimalistic version)

# Generating data for the funnel
stages = ["<b>Applications</b> 📤", "<b>Replies</b> 📩", "<b>Interviews</b> 🤝"]
values = [metrics['total_apps'], metrics['replied'], metrics['interviews']]
colors = ['#1D446D', '#5FB1C1', '#B34138']

initial_value = values[0]
percentages = [(v / initial_value) * 100 for v in values]

fig = go.Figure()

for i, (stage, value, pct, color) in enumerate(zip(stages, values, percentages, colors)):
    fig.add_trace(go.Bar(
        y=[stage],
        x=[value],
        orientation='h',
        # Shift the start of the bar to the left by half the value
        base=[-value/2 for _ in range(1)],
        marker=dict(
            color=color,
            cornerradius=15 # Adjusting rounding
        ),
        text=f"<b>{value} ({pct:.1f}%)</b>",
        textposition='inside',
        insidetextanchor='middle',
        insidetextfont=dict(size=14, color="white"), # if i > 0 else "black"),
        width=0.9,
        customdata=[[pct, value]],
        hovertemplate=(
            "<b>%{y}</b><br>" +
            "Value: %{customdata[1]}<br>" +  # Take the second value from customdata
            "Success Rate: %{customdata[0]:.1f}%<br>" +
            "<extra></extra>"
        ),
        showlegend=False
    ))

fig.update_layout(
    title_text='<b>JOB SEARCH CONVERSION</b>',
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        # Fix the range so that the funnel does not "jump"
        range=[-max(values)*0.6, max(values)*0.6]
    ),
    yaxis=dict(
        autorange="reversed",
        tickfont=dict(size=14),
        showline=False
    ),
    template="plotly_white",
    height=400,
    bargap=0.2
)

fig.show()

In [43]:
#@title [⏱️] Response Time Distribution (Wait Time)


# Filter only those rows that contain the response.
plot_data = df[df['response_time_days'] >= 0].dropna(subset=['response_time_days'])


fig = px.histogram(
    plot_data,
    x="response_time_days",
    nbins=20,
    title='<b>RESPONSE TIME DISTRIBUTION (Mean vs Median)</b>',
    labels={'response_time_days': 'Response time (days)'},
    template='plotly_white',
    color_discrete_sequence=['#5FB1C1']
)

# Add Mean
fig.add_vline(
    x=metrics['avg_res_time'],
    line_dash="dash",
    line_color="#1D446D",
    line_width=3,
    annotation_text=f"Mean: {metrics['avg_res_time']:.1f} days",
    annotation_position="top right",
    annotation_yshift=20,
    annotation_xshift=-20,
)

# Add Median
fig.add_vline(
    x=metrics['med_res_time'],
    line_dash="solid",
    line_color="#1D446D",
    line_width=5,
    annotation_text=f"Median: {metrics['med_res_time']:.0f} days",
    annotation_position="top left",
    annotation_yshift=20,
    annotation_xshift=30,
)

fig.update_layout(
    xaxis_title="Days from application",
    yaxis_title="Number of companies",
    bargap=0.1,
    barcornerradius=15
)

fig.show()

In [14]:
#@title [📈] Weekly Application Intensity

# Calculate the distribution: what % of the entire category falls on a specific day
# Now calculate the cross-table so that STATUSES are indexes
ct = pd.crosstab(df['response_status'], df['day_of_week'])
ct = ct.reindex(columns=days_order).fillna(0)

# Divide by the sum by row (by status) so that the horizontal sum is 100%
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

status_emoji_map = {
    'Active Interest': '🙂 Active Interest',
    'Rejection': '😞 Rejection',
    'No Feedback': '😐 No Feedback'
}

# Apply renaming to indexes for display
display_y = [status_emoji_map.get(status, status) for status in ct_pct.index]

# Preparing data for the bar (total number by day)
weekday_counts = df['date_applied'].dt.day_name().value_counts().reindex(days_order)

# Create a grid of 2 rows
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3], # Bigger hitmap, smaller bar
    vertical_spacing=0.1,
    subplot_titles=("<b>RESPONSE EFFECTIVENESS BY DAY</b>", "<b>TOTAL APPLICATION VOLUME</b>")
)

# ADD HEATMAP
fig.add_trace(
    go.Heatmap(
        z=ct_pct.values,
        x=ct_pct.columns,
        y=display_y,
        colorscale='darkmint',
        texttemplate="<b>%{z:.1f}%</b>",
        showscale=False,
        hovertemplate=(
            "<b>Day Applied:</b> %{x}<br>" +        # Day on X axis
            "<b>Outcome:</b> %{y}<br>" +            # Outcome on Y axis
            "<b>Share:</b> %{z:.1f}%" +             # Value
            "<extra></extra>"                       # Remove "trace 0"
        )
    ), row=1, col=1
)

# ADD BAR CHART
fig.add_trace(
    go.Bar(
        x=weekday_counts.index,
        y=weekday_counts.values,
        marker_color='#5FB1C1',
        text=weekday_counts.values,
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(
            color='white',
            size=14,
            family="Arial",
            weight='bold'
        ),
        name="Total Apps",
        hovertemplate="<b>%{x}</b><br>Total Applications: <b>%{y}</b><extra></extra>"
    ), row=2, col=1
)

# Additional adjustment for Bar axis
fig.update_xaxes(tickfont=dict(size=13), row=2, col=1)
fig.update_yaxes(title_text="Apps Count", row=2, col=1)

fig.update_layout(
    height=850,
    showlegend=False,
    template="plotly_white",
    title_text="<b>HOW WEEKDAY IMPACTS YOUR SUCCESS</b>",
    title_x=0.5,
    margin=dict(t=120, b=50, l=100, r=50),
    barcornerradius=15
)


fig.show()


In [17]:
#@title [💎] Success by Job Level (Senior vs Middle)

# Calculate statistics by grade
level_stats = df.groupby('level').agg(
    total=('position', 'count'),
    interviews=('status', lambda x: x.isin(['Interview', 'Test task']).sum())
).reset_index()

# Calculate the conversion rate within each grade
level_stats['conversion'] = (level_stats['interviews'] / level_stats['total'] * 100).round(1)

# Prepare text for display on the shares
# We show the grade name and its success rate
level_stats['label_text'] = (
    "<b>" + level_stats['level'] + "</b>" +
    "<br>Conv: " + level_stats['conversion'].astype(str) + "%"
)



fig = go.Figure(data=[go.Pie(
    labels=level_stats['level'],
    values=level_stats['total'],
    hole=.65,
    text=level_stats['label_text'],
    textinfo='text',
    insidetextorientation='horizontal',
    textposition='outside',

    # Setting the font of the text inside the segments
    insidetextfont=dict(size=18, color='black', family="Arial"),
    marker=dict(colors=['#1D446D', '#B34138', '#5FB1C1'] ,
                line=dict(color='#FFFFFF', width=2)),
    pull=[0.05, 0],
    hoverinfo='label+value+percent',
)])


total_all = level_stats['total'].sum()

fig.update_layout(
    title_text='<b>DISTRIBUTION OF RESPONSES AND THEIR SUCCESS BY LEVEL</b>',
    annotations=[dict(
        text=f'Total<br>{total_all}',
        x=0.5, y=0.5,
        font_size=22,
        showarrow=False,
        font_family="Arial"
    )],
    showlegend=False,
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5),
    margin=dict(t=50, b=50, l=10, r=10)
)

# fig.add_annotation(
#     dict(
#         x=0.5, y=-0.1,
#         xref="paper", yref="paper",
#         text=(
#             "<b>Insight:</b> Senior/Middle conversion is equal (8.5%), suggesting a strong Senior-level market fit."
#         ),
#         showarrow=False,
#         font=dict(size=16, color="#2c3e50"),
#         bgcolor="#ecf0f1",
#         bordercolor="#bdc3c7",
#         borderwidth=1,
#         borderpad=10
#     )
# )

fig.show()


# 💡 Key Search Insights

> **Search Period:** 146 Days | **Total Applications:** 141 | **Overall Conversion:** 8.5%

---

### 🚀 Strategic Takeaways

* **The "Monday Effect": High Volume, High Reward**

*Insight:* Monday is the most productive and effective day of the week, accounting for the peak in application volume (35 apps) and the highest interview conversion rate (>30%).

*Strategy:* Prioritize the beginning of the week as the primary "application window." Applications sent on Mondays likely land at the top of recruiters' inboxes, ensuring maximum visibility.

* **Conversion Consistency (Senior vs. Middle)**

*Insight:* There is a remarkable consistency in performance across levels; the interview conversion rate is quite close (8-9%) for both Middle and Senior positions.

*Takeaway:* My profile is equally competitive at both seniority levels. This justifies a strategic shift toward Senior roles, as they offer higher potential compensation and complexity for the same acquisition cost.

* **Response Dynamics & "Ghosting" Metrics**

*Insight:* While the average response time is 10.7 days, the median of 5 days indicates that most engaged companies react within the first business week.

*Takeaway:* If no feedback is received within 10–12 days, the probability of a positive outcome drops significantly. A "No Response" (Ghosting) rate of 32% is below market average, suggesting high-quality lead selection and a well-optimized CV.

* **The Mid-Week Slump (Wednesday)**

*Insight:* Wednesday appears to be a "blind spot" in the search cycle, showing the lowest weekday activity (15 apps) and the lowest conversion rate (~8%).

*Takeaway:* Efficiency drops mid-week. This day might be better utilized for "sharpening the saw"—updating the portfolio, networking, or deep-diving into specific company research—rather than high-volume outreach.